In [4]:
# CNN + BiLSTM + CTC — Full Training Notebook (LibriSpeech train/dev/test)

# Berikut notebook lengkap yang sudah **disesuaikan**:
# - otomatis load 3 split LibriSpeech
# - PyTorch Dataset & DataLoader
# - CNN + BiLSTM encoder
# - CTC loss
# - training + validation loop
# - checkpoint saving
# - logging sederhana (print + optional TensorBoard)


##############################################
# 1. Import
##############################################
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
from jiwer import wer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

##############################################
# 2. Paths (Kaggle)
##############################################
LIBRI_ROOT = "/kaggle/input/librispeech-clean/LibriSpeech"
TRAIN_DIR = os.path.join(LIBRI_ROOT, "train-clean-100")
DEV_DIR   = os.path.join(LIBRI_ROOT, "dev-clean")
TEST_DIR  = os.path.join(LIBRI_ROOT, "test-clean")








Device: cuda


In [5]:

##############################################
# 3. Text Tokenizer (char-level)
##############################################
class TextTransform:
    def __init__(self):
        chars = "' abcdefghijklmnopqrstuvwxyz"
        self.char2idx = {c:i for i,c in enumerate(chars)}
        self.idx2char = {i:c for c,i in self.char2idx.items()}

    def text_to_int(self, text):
        return torch.tensor([self.char2idx[c] for c in text])

    def int_to_text(self, labels):
        return ''.join([self.idx2char[i] for i in labels])

text_transform = TextTransform()



In [6]:

##############################################
# 4. LibriSpeech Dataset Wrapper
##############################################
class LibriDataset(Dataset):
    def __init__(self, root_dir):
        self.items = []
        for root, dirs, files in os.walk(root_dir):
            for f in files:
                if f.endswith(".trans.txt"):
                    trans_path = os.path.join(root, f)
                    with open(trans_path, 'r') as tf:
                        for line in tf:
                            utt, txt = line.strip().split(' ', 1)
                            txt = txt.lower()
                            flac_path = os.path.join(root, utt + ".flac")
                            if os.path.exists(flac_path):
                                self.items.append((flac_path, txt))

        self.mel = T.MelSpectrogram(sample_rate=16000, n_mels=80)
        self.db  = T.AmplitudeToDB()

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, txt = self.items[idx]
        audio, sr = torchaudio.load(path)
        if sr != 16000:
            audio = torchaudio.functional.resample(audio, sr, 16000)

        mel = self.db(self.mel(audio)).squeeze(0)  # [80, T]
        label = text_transform.text_to_int(txt)
        return mel.T, label  # -> (Time, Mel)



In [7]:

##############################################
# 5. Collate Function (handling variable lengths)
##############################################
def collate_fn(batch):
    feats = [b[0] for b in batch]
    labels = [b[1] for b in batch]

    feat_lengths = torch.tensor([f.shape[0] for f in feats])
    label_lengths = torch.tensor([l.shape[0] for l in labels])

    # pad feats
    feats_padded = nn.utils.rnn.pad_sequence(feats, batch_first=True)
    labels_padded = nn.utils.rnn.pad_sequence(labels, batch_first=True)

    return feats_padded, labels_padded, feat_lengths, label_lengths



In [9]:
##############################################
# 6. Model — CNN + BiLSTM + CTC
##############################################
class SpeechModel(nn.Module):
    def __init__(self, n_mels=80, hidden=256, num_layers=3, num_classes=30):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(n_mels, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.lstm = nn.LSTM(256, hidden, num_layers=num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_classes)

    def forward(self, x):  # x: [B, T, Mel]
        x = x.transpose(1,2)  # [B, Mel, T]
        x = self.cnn(x)
        x = x.transpose(1,2)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out


In [10]:

##############################################
# 7. Setup Datasets & Loaders
##############################################
train_ds = LibriDataset(TRAIN_DIR)
val_ds   = LibriDataset(DEV_DIR)
test_ds  = LibriDataset(TEST_DIR)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)


In [11]:

##############################################
# 8. Training Setup
##############################################
model = SpeechModel().to(DEVICE)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [12]:

##############################################
# 9. Training + Validation Loop
##############################################
EPOCHS = 5
best_val = 1e9
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0

    for feats, labels, feat_lens, label_lens in train_loader:
        feats = feats.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()

        logits = model(feats)  # [B, T, C]
        log_probs = nn.functional.log_softmax(logits, dim=-1)

        loss = criterion(log_probs.transpose(0,1), labels, feat_lens, label_lens)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for feats, labels, feat_lens, label_lens in val_loader:
            feats = feats.to(DEVICE)
            labels = labels.to(DEVICE)
            logits = model(feats)
            log_probs = nn.functional.log_softmax(logits, dim=-1)
            loss = criterion(log_probs.transpose(0,1), labels, feat_lens, label_lens)
            val_loss += loss.item()

    print(f"Epoch {epoch}: Train {train_loss/len(train_loader):.4f} | Val {val_loss/len(val_loader):.4f}")

    # save best checkpoint
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✔ Saved Best Model")


Epoch 1: Train 2.0970 | Val 1.3189
✔ Saved Best Model
Epoch 2: Train 1.0752 | Val 1.0554
✔ Saved Best Model
Epoch 3: Train 0.8879 | Val 0.9550
✔ Saved Best Model
Epoch 4: Train 0.8031 | Val 0.8619
✔ Saved Best Model
Epoch 5: Train 0.7637 | Val 0.8608
✔ Saved Best Model


In [15]:
##############################################
# 10. Simple Decoding + Test WER
##############################################
def greedy_decode(logits):
    probs = torch.argmax(logits, dim=-1).cpu().numpy()
    results = []
    for seq in probs:
        prev = -1
        out = []
        for p in seq:
            if p != prev and p != 0:
                out.append(p)
            prev = p
        results.append(out)
    return results

# Evaluate on test-clean
from torch.nn.functional import log_softmax

model.load_state_dict(torch.load("checkpoints/best_model.pth", map_location=DEVICE))
model.eval()

refs = []
preds = []

for mel, lbl, feat_l, lab_l in DataLoader(test_ds, batch_size=1, collate_fn=collate_fn):
    mel = mel.to(DEVICE)
    logits = model(mel)
    decoded = greedy_decode(logits)
    pred_text = text_transform.int_to_text(decoded[0])

    true = lbl[0][:lab_l[0]].tolist()
    true_text = text_transform.int_to_text(true)

    preds.append(pred_text)
    refs.append(true_text)

print("Test WER:", wer(refs, preds))

Test WER: 0.6459791539866099
